# Time series (PCMCI, PCMCI+, LPCMCI, Granger, VARMA-LiNGAM, longitudinal LiNGAM)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/time_series.ipynb)

Six methods find causes in data measured over time. PCMCI and Granger find the links from earlier
steps; PCMCI+ also finds links within one step, and LPCMCI allows series that are not measured.
VARMA-LiNGAM estimates the weights of the links within one step and from the step before.
Longitudinal LiNGAM takes a panel instead: the same units measured at a few time points. Each
returns a `TemporalStructure`, one graph per lag.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.6.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[numba,viz]==0.6.0" pandas

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.viz
from andrey.data import sample_scm

## Data

`sample_scm` draws one series of 1,000 time steps over 4 variables: a vector autoregression with
one lag, links within each step from a random DAG, and uniform noise. The drawing merges the lags
into one graph; a loop is a variable's dependence on its own past.

In [3]:
series = sample_scm(
    graph="erdos_renyi",
    functional="var",
    noise="uniform",
    d=4,
    n=1000,
    seed=1,
    density=1.0,
)
X, truth = series.data, series.graph
print(f"{X.shape[0]} time steps of {X.shape[1]} variables")
print(truth)
display(andrey.viz.draw(truth.summary_graph()))

1000 time steps of 4 variables
temporal  |  4 nodes  |  2 lags

  lag 0  |  2 edges (2 directed)
    2 -> 1
    2 -> 3

  lag 1  |  10 edges (10 directed)
    0 -> 0
    0 -> 1
    0 -> 3
    1 -> 1
    2 -> 1
    3 -> 1
    2 -> 2
    2 -> 3
    3 -> 2
    3 -> 3


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="240" fill="#fafbfc"/><path d="M 28.00 83.61 C 19.00 67.61 45.00 67.61 36.00 83.61" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="36.00,83.61 32.50,76.61 41.00,77.61" fill="#3a4658" stroke-linejoin="round"/><path d="M 44.97 97.51 Q 184.00 107.16 309.66 137.92" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.37,140.30 310.85,133.06 308.47,142.77" fill="#3a4658" stroke-linejoin="round"/><line x1="43.78" y1="102.12" x2="111.17" y2="133.64" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="120.22,137.88 113.28,129.11 109.05,138.17" fill="#3a4658" stroke-linejoin="round"/><path d="M 328.00 130.39 C 319.00 114.39 345.00 114.39 336.00 130.39" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="336.00,130.39 332.50,123.39 341.00,124.39" fill="#3a4658" stroke-linejoin="round"/><line x1="309.00" y1="143.39" x2="245.00" y2="143.39" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,143.39 309.00,138.39 309.00,148.39" fill="#3a4658" stroke-linejoin="round"/><path d="M 310.57 135.03 Q 232.00 104.39 144.11 138.66" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.89,138.66 312.39,130.37 308.76,139.69" fill="#3a4658" stroke-linejoin="round"/><path d="M 228.00 130.39 C 219.00 114.39 245.00 114.39 236.00 130.39" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="236.00,130.39 232.50,123.39 241.00,124.39" fill="#3a4658" stroke-linejoin="round"/><path d="M 219.48 139.88 Q 182.00 129.39 154.15 137.19" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="144.52,139.88 155.50,142.00 152.80,132.37" fill="#3a4658" stroke-linejoin="round"/><path d="M 144.52 146.89 Q 182.00 157.39 209.85 149.59" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.48,146.89 208.50,144.77 211.20,154.40" fill="#3a4658" stroke-linejoin="round"/><path d="M 128.00 130.39 C 119.00 114.39 145.00 114.39 136.00 130.39" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="136.00,130.39 132.50,123.39 141.00,124.39" fill="#3a4658" stroke-linejoin="round"/><circle cx="32.00" cy="96.61" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="96.61" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="332.00" cy="143.39" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="143.39" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="232.00" cy="143.39" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="143.39" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="132.00" cy="143.39" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="143.39" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text></svg>'

## Links from the past: PCMCI and Granger

PCMCI tests each lagged link with partial correlations, given the strongest earlier causes it
finds first; `tau_max` is the largest lag tested. Granger regresses each variable on the past
`maxlag` steps of every variable with a cross-validated lasso. Neither tests links within one step.

In [4]:
print(andrey.pcmci(X, tau_max=1), end="\n\n")
print(andrey.granger(X, maxlag=1))

pcmci  temporal  |  4 nodes  |  1 lags

  lag 1  |  10 edges (10 directed)
    0 -> 0
    0 -> 1
    0 -> 3
    1 -> 1
    2 -> 1
    3 -> 1
    2 -> 2
    2 -> 3
    3 -> 2
    3 -> 3

  p_values: a list of 1, too long to print; read metadata['p_values']
  statistics: a list of 1, too long to print; read metadata['statistics']
  parents: [[[0, 1], [1, 1]], [[3, 1], [0, 1], [2, 1]], [[3, 1], [2, 1]], [[3, 1], [2, 1], [0, 1]]]

granger  temporal  |  4 nodes  |  1 lags

  lag 1  |  13 edges (13 directed)
    0 -> 0
    0 -> 1
    1 -> 0
    0 -> 2
    0 -> 3
    1 -> 1
    2 -> 1
    1 -> 3
    3 -> 1
    2 -> 2
    2 -> 3
    3 -> 2
    3 -> 3


## Links within a step: PCMCI+ and LPCMCI

PCMCI+ adds the same-step links and orients them where the data allow; lag 0 is a CPDAG. LPCMCI
allows hidden series, so it returns a partial ancestral graph (PAG): a circle marks an end the data
leave open.

In [5]:
print(andrey.pcmciplus(X, tau_max=1), end="\n\n")
print(andrey.lpcmci(X, tau_max=1))

pcmciplus  temporal  |  4 nodes  |  2 lags

  lag 0  |  3 edges (3 directed)
    1 -> 2
    3 -> 1
    2 -> 3

  lag 1  |  7 edges (7 directed)
    0 -> 0
    0 -> 3
    3 -> 1
    2 -> 2
    2 -> 3
    3 -> 2
    3 -> 3

  p_values: a list of 2, too long to print; read metadata['p_values']
  statistics: a list of 2, too long to print; read metadata['statistics']
  parents: [[[0, 1]], [[3, 1], [0, 1]], [[3, 1], [2, 1]], [[3, 1], [2, 1], [0, 1]]]
  conflicts: []
  ambiguous_triples: [[2, 1, 3, 1]]

lpcmci  temporal  |  4 nodes  |  2 lags

  lag 0  |  2 edges (1 directed, 1 partially_directed)
    2 -> 1
    2 o-> 3

  lag 1  |  0 edges (no edges)

  links: a list of 2, too long to print; read metadata['links']
  p_values: a list of 2, too long to print; read metadata['p_values']
  statistics: a list of 2, too long to print; read metadata['statistics']
  parents: [[], [[2, 0]], [], []]
  conflicts: []


## Weights: VARMA-LiNGAM

VARMA-LiNGAM uses the non-Gaussian noise to order the variables within a step, and estimates the
weight of every link. `prune=True` sets small weights to zero.

In [6]:
out = andrey.varma_lingam(X, order=(1, 1), prune=True)
print(out, end="\n\n")
print("lag 0 weights, row = cause:")
print(np.round(out.structure.lag_weights[0], 2))

varmalingam  temporal  |  4 nodes  |  2 lags

  lag 0  |  2 edges (2 directed)
    2 -> 1
    2 -> 3

  lag 1  |  8 edges (8 directed)
    0 -> 0
    0 -> 1
    0 -> 3
    1 -> 1
    3 -> 1
    2 -> 2
    3 -> 2
    3 -> 3

  order: 2 -> 1 -> 0 -> 3

lag 0 weights, row = cause:
[[ 0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.  ]
 [ 0.   -1.38  0.    1.19]
 [ 0.    0.    0.    0.  ]]


## Compare the links

Each method's links at lags 0 and 1 beside the true ones, a variable's dependence on its own past
included. A same-step link counts once, whatever its direction; a lagged link counts with its
direction. LPCMCI is left out: its lagged graphs keep only the links it orients as causes, so its
PAG above is the answer to read.

In [7]:
def links(structure):
    found = set()
    for lag in (0, 1):
        if lag not in structure.lags:
            continue
        for i, j, _ in structure.lag(lag).oriented_edges():
            found.add((min(i, j), max(i, j), 0) if lag == 0 else (i, j, 1))
    return found


true_links = links(truth)
fits = {
    "PCMCI": andrey.pcmci(X, tau_max=1),
    "Granger": andrey.granger(X, maxlag=1),
    "PCMCI+": andrey.pcmciplus(X, tau_max=1),
    "VARMA-LiNGAM": andrey.varma_lingam(X, order=(1, 1), prune=True),
}
rows = {}
for name, fit in fits.items():
    found = links(fit.structure)
    rows[name] = {
        "true links found": f"{len(found & true_links)} of {len(true_links)}",
        "extra links": len(found - true_links),
    }
pd.DataFrame(rows).T

,true links found,extra links
PCMCI,10 of 12,0
Granger,10 of 12,3
PCMCI+,9 of 12,1
VARMA-LiNGAM,10 of 12,0


## Panels: longitudinal LiNGAM

Longitudinal LiNGAM takes one table per time point, the same units in the same rows. Here 500 units
are measured at 3 time points; within each, `x0 -> x1 -> x2`, and each variable depends on its own
value at the time point before.

In [8]:
rng = np.random.default_rng(0)
b0 = np.array([[0.0, 0.0, 0.0], [0.8, 0.0, 0.0], [0.0, -0.7, 0.0]])  # row = effect
mix = np.linalg.inv(np.eye(3) - b0)
panel = [rng.uniform(-1, 1, (500, 3)) @ mix.T]
for _ in range(2):
    panel.append((0.5 * panel[-1] + rng.uniform(-1, 1, (500, 3))) @ mix.T)

out = andrey.longitudinal_lingam(panel, n_lags=1)
print(out, end="\n\n")
print("lag 0 at the last time point:", out.structure.lag(0).oriented_edges())

longitudinallingam  temporal  |  3 nodes  |  2 lags

  lag 0  |  2 edges (2 directed)
    0 -> 1
    1 -> 2

  lag 1  |  9 edges (9 directed)
    0 -> 0
    0 -> 1
    1 -> 0
    0 -> 2
    2 -> 0
    1 -> 1
    1 -> 2
    2 -> 1
    2 -> 2

  causal_orders: [[], [0, 1, 2], [0, 1, 2]]

lag 0 at the last time point: [(0, 1, 'directed'), (1, 2, 'directed')]


The [release cards](https://andrey.abel.ai/docs/benchmarks.html) compare each method with tigramite,
lingam, or causal-learn on the simulations of the methods' papers.

## Use your own data

Pass a table with one row per time step, such as a DataFrame whose column names become the graph's
labels, or change `seed` and `n` above and rerun.